In [1]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

In [2]:
df = pd.read_csv("credit_card_fraud_dataset.csv")

print("First 5 Records")
print(df.head())

print("\nDataset Shape:", df.shape)

First 5 Records
   TransactionID             TransactionDate   Amount  MerchantID  \
0              1  2024-04-03 14:15:35.462794  4189.27         688   
1              2  2024-03-19 13:20:35.462824  2659.71         109   
2              3  2024-01-08 10:08:35.462834   784.00         394   
3              4  2024-04-13 23:50:35.462850  3514.40         944   
4              5  2024-07-12 18:51:35.462858   369.07         475   

  TransactionType      Location  IsFraud  
0          refund   San Antonio        0  
1          refund        Dallas        0  
2        purchase      New York        0  
3        purchase  Philadelphia        0  
4        purchase       Phoenix        0  

Dataset Shape: (100000, 7)


In [3]:
print("\nMissing Values")
print(df.isnull().sum())


Missing Values
TransactionID      0
TransactionDate    0
Amount             0
MerchantID         0
TransactionType    0
Location           0
IsFraud            0
dtype: int64


In [4]:
df.drop_duplicates(inplace=True)

In [5]:
df["TransactionDate"] = pd.to_datetime(df["TransactionDate"])

df["Year"] = df["TransactionDate"].dt.year
df["Month"] = df["TransactionDate"].dt.month
df["Day"] = df["TransactionDate"].dt.day
df["Hour"] = df["TransactionDate"].dt.hour

df.drop("TransactionDate", axis=1, inplace=True)


In [6]:
encoder = LabelEncoder()

categorical = ["MerchantID", "TransactionType", "Location"]

for col in categorical:
    df[col] = encoder.fit_transform(df[col])


In [7]:
X = df.drop("IsFraud", axis=1)
y = df["IsFraud"]

In [8]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

In [9]:
model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

model.fit(X_train, y_train)


RandomForestClassifier(random_state=42)

In [10]:
prediction = model.predict(X_test)


In [11]:
print("\nAccuracy")
print(accuracy_score(y_test, prediction))

print("\nConfusion Matrix")
print(confusion_matrix(y_test, prediction))

print("\nClassification Report")
print(classification_report(y_test, prediction))


Accuracy
0.98935

Confusion Matrix
[[19787     0]
 [  213     0]]

Classification Report
              precision    recall  f1-score   support

           0       0.99      1.00      0.99     19787
           1       0.00      0.00      0.00       213

    accuracy                           0.99     20000
   macro avg       0.49      0.50      0.50     20000
weighted avg       0.98      0.99      0.98     20000



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [12]:
importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": model.feature_importances_
})

importance = importance.sort_values(
    by="Importance",
    ascending=False
)

print("\nFeature Importance")
print(importance)


Feature Importance
           Feature  Importance
1           Amount    0.239234
0    TransactionID    0.233831
2       MerchantID    0.197875
7              Day    0.093285
8             Hour    0.081623
4         Location    0.064998
6            Month    0.061094
3  TransactionType    0.020607
5             Year    0.007453


In [13]:
sample = pd.DataFrame({
    "TransactionID":[100001],
    "Amount":[15000],
    "MerchantID":[10],
    "TransactionType":[1],
    "Location":[5],
    "Year":[2024],
    "Month":[8],
    "Day":[10],
    "Hour":[14]
})

result = model.predict(sample)

if result[0] == 1:
    print("\nFraudulent Transaction")
else:
    print("\nGenuine Transaction")


Genuine Transaction
